# Customer Churn Prediction

Practice activity from the Microsoft Foundations of AI and Machine Learning course on Coursera. The business scenario is a telecom company that wants to predict which customers are likely to cancel their service so the marketing team can step in with retention offers before it is too late.

I picked **Scikit-learn (RandomForestClassifier)** for this one. The reasoning:

- The data is tabular and small (a handful of features like tenure, monthly charges, contract type), which is the natural home of classical ML.
- The CRM integration requirement says the backend is Python, and Scikit-learn models pickle into a single file and load with one line.
- Random Forest gives solid baseline accuracy with feature importance baked in, which helps the marketing team understand *why* a customer was flagged.
- For real-time inference on millions of rows, a pruned Random Forest predicts in milliseconds.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import joblib
import warnings
warnings.filterwarnings("ignore")

## 2. Load the dataset

This is the tiny educational CSV provided by the course (5 rows). Real production data would obviously have millions of customers, but the workflow stays the same.

In [2]:
data = pd.read_csv('customer_churn.csv')
print(data.head())
print()
print(data.info())

   CustomerID  Tenure  MonthlyCharges  TotalCharges        Contract  \
0        1001       5            70.0         350.0  Month-to-month   
1        1002      10            85.5         850.5        Two year   
2        1003       3            55.3         165.9        One year   
3        1004       8            90.0         720.0  Month-to-month   
4        1005       2            65.2         130.4        One year   

      PaymentMethod  Churn  
0  Electronic check      1  
1      Mailed check      0  
2  Electronic check      1  
3       Credit card      0  
4  Electronic check      1  

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   CustomerID      5 non-null      int64  
 1   Tenure          5 non-null      int64  
 2   MonthlyCharges  5 non-null      float64
 3   TotalCharges    5 non-null      float64
 4   Contract        5 non-n

## 3. Preprocess the data

Drop the CustomerID column (no signal there), drop any rows with missing values, one-hot encode the categorical columns (Contract, PaymentMethod), then split into 80/20 train/test.

In [3]:
data = data.drop(columns=['CustomerID'])
data = data.dropna()
data = pd.get_dummies(data, drop_first=True)

X = data.drop('Churn', axis=1)
y = data['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print()
print('Features:', list(X.columns))

Train shape: (4, 7)
Test shape: (1, 7)

Features: ['Tenure', 'MonthlyCharges', 'TotalCharges', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


## 4. Build and train the model

Random Forest with 100 trees. Random state set so results are reproducible.

In [4]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print("Training complete.")

Training complete.


## 5. Evaluate on the test set

Accuracy is the headline metric, but precision/recall/F1 matter more for churn since the cost of false negatives (missed churners) is higher than false positives. With only 5 rows of data these numbers are not meaningful, but the workflow is the same with real data.

In [5]:
predictions = model.predict(X_test)
acc = accuracy_score(y_test, predictions)
print(f'Test accuracy: {acc:.4f}')
print()
print('Classification report:')
print(classification_report(y_test, predictions, zero_division=0))

Test accuracy: 1.0000

Classification report:


              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1

    accuracy                           1.00         1
   macro avg       1.00      1.00      1.00         1
weighted avg       1.00      1.00      1.00         1



## 6. Optimize for deployment

For production, the goal is to make the model lighter and faster without giving up much accuracy. A few standard tricks:

- Cap **max_depth** so trees do not memorize the training set.
- Limit **max_features** to a square root of the total number of features, which speeds up inference.
- Persist the model with **joblib** for a single-file artifact that the CRM backend can load with one call.

In [6]:
pruned_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=10,
    max_features='sqrt'
)
pruned_model.fit(X_train, y_train)
pruned_predictions = pruned_model.predict(X_test)
pruned_acc = accuracy_score(y_test, pruned_predictions)
print(f'Pruned test accuracy: {pruned_acc:.4f}')
print(f'Original test accuracy: {acc:.4f}')

Pruned test accuracy: 1.0000
Original test accuracy: 1.0000


## 7. Save the model for deployment

Persist the pruned model to a single .pkl file. To load it in the CRM backend later:

```python
import joblib
model = joblib.load("churn_model.pkl")
predictions = model.predict(customer_features)
```

In [7]:
joblib.dump(pruned_model, 'churn_model.pkl')
print('Saved churn_model.pkl')

# Sanity check: reload and predict
reloaded = joblib.load('churn_model.pkl')
reloaded_preds = reloaded.predict(X_test)
print('Reloaded model matches:', np.array_equal(reloaded_preds, pruned_predictions))

Saved churn_model.pkl


Reloaded model matches: True


## 8. Feature importance

Quick look at which features the Random Forest is leaning on. With a real dataset this helps the marketing team understand churn drivers and design targeted retention offers.

In [8]:
importances = pd.Series(pruned_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print('Feature importance ranking:')
print(importances)

Feature importance ranking:
Tenure                            0.255556
PaymentMethod_Electronic check    0.242857
MonthlyCharges                    0.228571
TotalCharges                      0.209524
Contract_One year                 0.063492
Contract_Two year                 0.000000
PaymentMethod_Mailed check        0.000000
dtype: float64


## Reflection

A few things worth noting:

1. **Dataset size is the bottleneck here.** The course CSV is 5 rows, which gives a 4 train / 1 test split. Any metric calculated on a single test row is essentially meaningless, but the *pipeline* is identical to what I would run on a million row dataset.
2. **Scikit-learn was the right pick for this problem.** Tabular structured data, fast iteration, dead simple deployment story (one .pkl file the CRM can pickle.load). TensorFlow or PyTorch would have been overkill.
3. **Pruning works.** Capping max_depth=10 and max_features=sqrt keeps the model lean, which matters when the CRM has to serve predictions in near real time for millions of customers.
4. **Next step for real deployment:** wrap the model in a FastAPI or Azure App Service endpoint, log predictions for monitoring, schedule periodic retraining on fresh data, and add a drift detection job.